In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-2: Preferences, reward models and RLHF

**RL for LLMs, session 2.**


## What this session teaches

A reward from people usually starts as comparisons: a person sees two answers to one prompt and says which is better. This is easier and more consistent than giving each answer a score.

RL needs a number, not a comparison. So a **reward model** is trained first: a language model that gives a reward to a prompt and an answer, trained so that the preferred answer of each pair gets the higher reward.

**RLHF** (reinforcement learning from human feedback) is three stages: SFT tunes the model on demonstrations, prompts with good answers people wrote; a reward model is trained on comparisons; then RL tunes the SFT model, now called the **policy**, to earn high rewards from the reward model. <!-- long-ok -->

The reward model is only an estimate of what people prefer. Pushed hard, the policy finds answers that score high but that people would not prefer. This is **over-optimization**.

A penalty keeps the policy's probabilities near those of a frozen copy of the SFT model, called the **reference model**. It limits over-optimization, but does not cure it.

The question to carry: how far can a learned reward model be trusted?

By the end you can

- say what each stage of RLHF trains, and on what data
- say how comparisons become a reward model
- say why pushing on a reward model goes wrong, and what the penalty does

**Where this session fits: the topics of the series, in order**

<img src="images/rllm2v2_strip.png" width=88%>

- this session starts topic 2: a reward from people's comparisons


## Setting and words

**Why this slide?** The reward-model slides use these words; each is said by what it does.

| word | what it means here |
|:--|:--|
| **answer** (also called a **completion**) | the text the model writes for a prompt; formulas write $\y$ |
| **$\x$, $\y^+$, $\y^-$** | one comparison: the prompt, the answer people preferred, the other answer |
| **$\psi$** | the numbers that are trained in the reward model (the policy's are $\theta$, the Intro critic's are $\phi$) |
| **frozen** | not updated during a stage |
| **policy** | the model that RL trains: it starts as the SFT model |
| **demonstration** | a prompt with a good answer a person wrote; SFT trains on these |


## RLHF on one page: three stages

**Why this section?** Each stage trains a different model on different data. Seeing all three first shows where every later slide sits.

> **RLHF has three stages: SFT, a reward model trained on comparisons, then RL against that reward model.**

<img src="images/rllm2v2_pipeline.png" width=60%>

- a **demonstration** is a prompt with a good answer someone wrote (RLLM-1)
- SFT trains on demonstrations; the reward model is trained separately, on comparisons
- the sections follow the stages in order: stage 1 (SFT), stage 2 (reward model), stage 3 (RL)


## Stage 1: SFT

**You are here:** stage **1 SFT** → 2 reward model → 3 RL

**Why this section?** The first stage fixes the model that everything else starts from.

> **SFT tunes the pretrained model on demonstrations: prompts with good answers people wrote, of the kind of task the assistant is for. The result is the SFT model.**

<img src="images/rllm2v2_pipeline_bar_1.png" width=56%>

- RL starts from the SFT model, which is then called the **policy**
- a frozen copy of the SFT model becomes the **reference model** of the last stage
- the SFT model also writes the answers that people compare in the next stage


## Stage 2: comparisons

<img src="images/rllm2v2_pipeline_bar_2.png" width=56%>

**You are here:** 1 SFT → stage **2 reward model** → 3 RL

**Why this section?** RL needs a reward for every answer, but people find it easier to say which of two answers is better.

> **Preference data: a prompt and two answers, with the person's choice. It says better, not by how much.**

<img src="images/rllm2v2_preference.png" width=52%>

- the two answers usually come from the SFT model, sampled twice for each prompt
- choosing between two is easier and more consistent than scoring one (our interpretation)
- the cost: every pair needs a paid person to read two answers
- so comparisons by people are slow and costly
- RLLM-3 asks whether a model can make them

**The reward model: a network that gives a reward to an answer**

> **The reward model is a language model whose next-token output is replaced by a single output: the reward of the answer.**

<img src="images/rllm2v2_reward_model.png" width=62%>

- it is trained on comparisons: several answers per prompt, ranked by people
- $m$ ranked answers give $m(m-1)/2$ pairs
- the next two slides say how: first the assumption that links rewards to comparisons, then the loss
- it does not know the truth, only what people tended to prefer
- reported weaknesses: a preference for long answers, and reliability only near the answers it was trained on

**From comparisons to a reward: Bradley-Terry**

> **Bradley-Terry converts the gap between the rewards of two answers into the chance that one is preferred to the other. It says: the chance that $\y^+$ is preferred is the sigmoid of the gap.**

<img src="images/rllm2v2_sigmoid.png" width=38%>

- **why:** we never see rewards, only winners, so we need a model of how a reward gap makes a win likely
- the picture: rewards 2.0 and 0.5 give a gap of 1.5, so a chance of 0.82 that $\y^+$ is preferred
- the reversed pair has a gap of $-1.5$ and a chance of 0.18
- a gap of 0 gives 0.5; a bigger gap gives a more confident preference
- only the gap matters, so a reward is known only up to a constant for each prompt
- it is an assumption, not a fact about people
- it assumes one pool of raters, and that if A beats B and B beats C then A beats C
- it is used only to train the reward model

**Training the reward model: the loss**

> **Train the reward model, a network with parameters $\psi$, to give $\y^+$ a higher reward than $\y^-$. The loss function, averaged over comparisons, is $-\log \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$: cross-entropy for the observed winner.**

<img src="images/rllm2v2_loss.png" width=44%>

- this is ordinary binary classification: "which of the two answers won?"
- the reward gap is the **logit**, the raw number before the sigmoid turns it into a chance
- the loss reaches zero only as the gap grows without limit, so on a pair that people got wrong, training keeps pushing (our interpretation)
- the example gap of 1.5 costs 0.20; the reversed pair costs 1.70


## Stage 3: RL against the reward model

<img src="images/rllm2v2_pipeline_bar_3.png" width=56%>

**You are here:** 1 SFT → 2 reward model → stage **3 RL**

**Why this section?** The reward model exists and is frozen. Now it is used.

> **The reward model scores a whole answer, so the reward arrives once, at the last token. How it reaches each token is the subject of RLLM-5.**

<img src="images/rllm2v2_one_reward.png" width=62%>

- the policy samples answers, the reward model scores them, and an optimizer (PPO or GRPO) updates the policy
- the question of which token earned how much of the reward is **credit assignment**

**The problem: over-optimization**

**Why this slide?** The reward model is an estimate. The harder the policy chases it, the more it finds answers where the estimate is wrong.

> **Pushed hard, the policy finds answers the reward model over-rates, and measured quality falls: over-optimization.**

<img src="images/rllm2_overoptimization.png" width=56%>

- schematic, invented curves; the shape is the one [Gao et al. (2022, ICML 2023)](https://arxiv.org/abs/2210.10760) measured
- they used a fixed **gold** reward model, a large model treated as the truth, in place of people
- the reward the policy trains on keeps rising; the gold reward peaks and then falls
- **reward hacking** is one way it happens: an answer exploits a blind spot of the reward model

**The penalty: a reference model**

**Why this slide?** The reward model is reliable only near the answers it was trained on, so we keep the policy near its start.

> **A penalty holds the policy's probabilities near those of a frozen reference model. It limits over-optimization but does not cure it.**

<img src="images/rllm2v2_penalty.png" width=48%>

- the policy maximizes the reward minus a coefficient $\beta$ times a **KL divergence**
- the KL divergence measures how far its probabilities are from the reference model's
- it limits: the policy stays near answers the reward model saw
- it does not cure: the reference is only an SFT model, and the reward model can be wrong near it
- $\beta$ has no formula; it is tuned by trial


# What we established, and what it sets up

- **Established**
    - people's comparisons are easier to collect than scores
    - a reward model is trained on them with Bradley-Terry and a classification loss
    - RLHF has three stages: SFT, a reward model, then RL against the frozen reward model
    - the reward arrives once, at the last token
    - pushing hard on a reward model over-optimizes
    - a penalty toward the reference model, a frozen copy of the SFT model, limits this but does not cure it
- **Sets up**
    - **the limit:** a reward model is an estimate, and comparisons by people are slow and costly
    - next session, RLLM-3: can a model make the comparisons instead, guided by written principles?


## Where the penalty arrives, and reading $\beta$ (aside)

- in PPO the penalty arrives at every token, as $-\beta\, d$ with $d = \log \dfrac{\pi_\theta(\y_{\tt} \mid \x, \y_{<\tt})}{\pi_{\text{ref}}(\y_{\tt} \mid \x, \y_{<\tt})}$ (Intro session C1); in GRPO it is a loss term instead
- $\beta$ is the coefficient: large stays close, small chases the reward
- a known result: the best policy under this objective is proportional to $\pi_{\text{ref}}(\y \mid \x)\, e^{\rew/\beta}$; RLLM-10 (DPO) starts from it
- the penalty may also protect what the base model already knows (our interpretation)
- Gao et al. found that at a given KL the penalty did not measurably improve the gold reward in their setup; it acted like stopping earlier

## An example preference dataset, and origins (aside)

- [UltraFeedback (binarized)](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized): prompt, chosen, rejected; labeled by a model (GPT-4), not by people
- [Christiano et al. 2017](https://arxiv.org/abs/1706.03741), [Ziegler et al. 2019](https://arxiv.org/abs/1909.08593), [Stiennon et al. 2020](https://arxiv.org/abs/2009.01325), [Ouyang et al. 2022](https://arxiv.org/abs/2203.02155) (InstructGPT)
